# ⚡ LLM Quantization Tradeoff Analyzer
**Target:** AMD/NVIDIA inference optimization roles  
**Measures:** TTFT · TPS · VRAM · Semantic Quality across FP16 / INT8 / INT4

### Setup
- Enable GPU: Runtime → Change runtime type → GPU (T4 preferred)
- Run all cells in order
- Gradio UI launches in the last cell

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers bitsandbytes accelerate gradio sentence-transformers pandas matplotlib seaborn
print('✓ Dependencies installed')

In [ ]:
# Cell 2: Check GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Cell 3: Upload quantization_benchmark.py (or paste directly)
# Option A: if you uploaded the file
# exec(open('quantization_benchmark.py').read())

# Option B: clone from your repo
# !git clone https://github.com/YOUR_USERNAME/llm-quant-analyzer
# %cd llm-quant-analyzer

# For now, we'll paste the key classes inline in the next cells
print('Ready to run inline version')

In [ ]:
# Cell 4: Core benchmark code (inline)
# Paste the full contents of quantization_benchmark.py here
# OR do: exec(open('/kaggle/input/your-dataset/quantization_benchmark.py').read())

import os, gc, json, time, warnings
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')

# --- Paste ModelLoader, InferenceRunner, MetricsCollector, QualityEvaluator, QuantizationBenchmark here ---
# Or run: exec(open('quantization_benchmark.py').read())
print('Classes loaded. Proceed to Cell 5.')

In [ ]:
# Cell 5: ⚙️ Configure and run benchmark
# Adjust model and settings here

CONFIG = dict(
    model_key   = 'TinyLlama-1.1B',    # Options: TinyLlama-1.1B | Phi-3-mini | Qwen2.5-3B
    precisions  = ['FP16', 'INT8', 'INT4'],
    n_perf_runs = 5,                    # Inference passes per precision (more = more stable)
    max_quality_per_category = 3,       # Prompts per category (3-5 recommended)
    output_dir  = './benchmark_results',
)

print('Config:', CONFIG)
print()
print('⚠️  This will take ~15-30 min for TinyLlama on T4.')
print('    Phi-3-mini: ~25-45 min')
print('    Qwen2.5-3B: ~35-60 min')

In [ ]:
# Cell 6: 🚀 Run the benchmark (takes ~20-40 min)
bench = QuantizationBenchmark(**CONFIG)
perf_results, quality_results = bench.run()
print('\n✅ Benchmark complete!')

In [ ]:
# Cell 7: Quick results check
df_perf = pd.DataFrame(perf_results)
print('Performance Results:')
print(df_perf[['precision','vram_mb','ttft_ms','tps','total_latency_ms']].to_string(index=False))

df_qual = pd.DataFrame([{k:v for k,v in r.items() if k != 'response'} for r in quality_results])
print('\nQuality Summary by Precision:')
print(df_qual.groupby('precision')[['semantic_score','formatting_ok','is_complete']].mean().round(3))

In [ ]:
# Cell 8: 📊 Launch Gradio Dashboard
# Upload gradio_app.py first, then:
# exec(open('gradio_app.py').read())

# Or run inline:
import gradio as gr

# build_app is defined in gradio_app.py — paste or import it here
# then:
app = build_app(results_dir='./benchmark_results', demo_mode=False)
app.launch(share=True)   # share=True gives a public URL on Kaggle

In [ ]:
# Cell 9: 💾 Save results for download
import shutil
shutil.make_archive('quantization_results', 'zip', './benchmark_results')
print('Results archived to quantization_results.zip')
print('Download from: Files panel → quantization_results.zip')

## Interpreting Results

### The Key Tradeoffs

| Metric | FP16 | INT8 | INT4 |
|--------|------|------|------|
| Memory | Highest | ~50% of FP16 | ~25% of FP16 |
| TTFT | Baseline | Faster | Fastest |
| TPS | Baseline | +20-40% | +40-60% |
| Quality | Best | ~Same | Small drop |

### Why latency gains < VRAM gains for INT4
INT4 weights must be **dequantized back to FP16** before matrix multiplication (cuBLAS/cuDNN operate in FP16). This adds ALU load and means inference becomes **partially compute-bound** even as memory usage drops dramatically.

### Interview-ready insight
> *"The memory savings from INT4 are nearly 4× vs FP16, but throughput improvement is only 40-60%. This gap exists because dequantization overhead shifts the bottleneck from memory bandwidth to compute, partially negating the savings on the compute pipeline."*